## Итоговые результаты

| Вариант | OOF AUC | Public LB |
|---|---|---|
| одиночный CatBoost | ~0.834 | — |
| ансамбль 7 моделей (простое среднее) | 0.8367 | 0.7667 |
| **ансамбль + seed-bagging** | 0.8376 | **0.7672 |

Решение — взвешенный ансамбль градиентного бустинга трёх типов. Главные находки:
разнообразие моделей важнее тюнинга, простое среднее устойчивее взвешенного,
stratified CV обобщает лучше хронологической несмотря на временной сдвиг данных.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

SEED, N_FOLDS = 42, 5
ID_COL, TARGET = "front_id", "target_value"
CAT = ["db_group_last", "fl_adminarea"]
DATA_DIR = "."  # csv лежат рядом с ноутбуком

train = pd.read_csv(f"{DATA_DIR}/train_apps.csv")
test = pd.read_csv(f"{DATA_DIR}/test_apps.csv")
print(train.shape, test.shape)

## Разведка данных

Сильный дисбаланс (6% согласий), много пропусков (до 96% в столбце — означают
отсутствие продукта у клиента), и тест продолжает train по времени с растущей долей
согласий. Из-за сдвига порядковый день стал одним из главных признаков.

In [ ]:
print("доля согласий:", round(train[TARGET].mean(), 4))
print("\nпропуски (топ):")
print(train.isna().mean().sort_values(ascending=False).head(6).round(3))

trd, ted = pd.to_datetime(train["decision_day"]), pd.to_datetime(test["decision_day"])
print("\ntrain:", trd.min().date(), "->", trd.max().date())
print("test :", ted.min().date(), "->", ted.max().date())
dec = pd.qcut((trd - trd.min()).dt.days, 5, labels=False)
print("\nдоля согласий по периодам:")
print(train.assign(b=dec).groupby("b")[TARGET].mean().round(4))

## Признаки

Работаем на объединённом train+test, чтобы кодирования были согласованы. К исходным
столбцам добавляем: временные, флаги пропусков, доменные interactions и target encoding.

In [ ]:
y = train[TARGET].astype(int).copy()
train["is_train"], test["is_train"] = 1, 0
test[TARGET] = np.nan
df = pd.concat([train, test], ignore_index=True)

# временные признаки; порядковый день ловит общий тренд
dt = pd.to_datetime(df["decision_day"], errors="coerce")
df["dt_month"], df["dt_dayofweek"] = dt.dt.month, dt.dt.dayofweek
df["dt_day"], df["dt_quarter"] = dt.dt.day, dt.dt.quarter
df["dt_ordinal"] = (dt - dt.min()).dt.days
date_feats = ["dt_month", "dt_dayofweek", "dt_day", "dt_quarter", "dt_ordinal"]
df.drop(columns=["decision_day"], inplace=True)

# категории — label encoding
for c in CAT:
    df[c] = LabelEncoder().fit_transform(df[c].fillna("__MISSING__").astype(str))

In [ ]:
# флаги пропусков по самым «дырявым» столбцам + общий счётчик
useful = ["overdraft_app_term_max_360", "loan_rev_max_start_non_fin", "loan_rev_min_start_fin",
          "days_from_authperson_registration", "app_term_mean_360", "sum_deb_ul_90", "corp_credit_products"]
miss = []
for c in useful:
    df[f"{c}_is_missing"] = df[c].isnull().astype("int8")
    miss.append(f"{c}_is_missing")
df["total_missing_count"] = df[[c for c in df.columns if c not in [ID_COL, TARGET, "is_train"]]].isnull().sum(axis=1)
miss.append("total_missing_count")

In [ ]:
# interactions: произведения, отношения, разности по предметной логике
interactions = {
    "loan_rev_x_cnt_cred": df["loan_rev_max_start_non_fin"] * df["cnt_cred_loan_90"],
    "loan_rev_x_overdraft_max": df["loan_rev_max_start_non_fin"] * df["overdraft_limit_max"],
    "loan_rev_x_loan_amount": df["loan_rev_max_start_non_fin"] * df["loan_amount_last"],
    "loan_ops_total": df["cnt_deb_loan_90"] + df["cnt_cred_loan_90"],
    "loan_ops_ratio_safe": df["cnt_deb_loan_90"] / df["cnt_cred_loan_90"].clip(lower=0.01),
    "overdraft_limit_range": df["overdraft_limit_max"] - df["overdraft_limit_min"],
    "loan_to_overdraft_max": df["loan_amount_last"] / df["overdraft_limit_max"].clip(lower=0.01),
    "rate_ratio": df["offered_rate"] / df["cb_rate"].clip(lower=0.001),
    "deb_ul_ip_trend": df["cnt_deb_ul_ip_90"] - df["cnt_deb_ul_ip_30"],
    "deb_ul_90_to_30_ratio": df["cnt_deb_ul_ip_90"] / df["cnt_deb_ul_ip_30"].clip(lower=0.01),
    "loan_rev_range": df["loan_rev_max_start_non_fin"] - df["loan_rev_min_start_fin"],
    "corp_engagement": df["count_all_corp_dashboard_events"] * df["p75_time_spent_minutes"],
    "deb_ul_diff_90_30": df["sum_deb_ul_90"] - df["cnt_deb_ul_ip_30"],
    "loan_deb_cred_diff": df["cnt_deb_loan_90"] - df["cnt_cred_loan_90"],
    "loan_deb_cred_ratio": df["cnt_deb_loan_90"] / df["cnt_cred_loan_90"].clip(lower=0.01),
}
for name, series in interactions.items():
    df[name] = series
inter = list(interactions)
len(inter)

In [ ]:
# out-of-fold target encoding — без утечки таргета
def target_encode_cv(col, smoothing=50):
    gmean = df.loc[df["is_train"] == 1, TARGET].mean()
    df[f"{col}_te"] = np.nan
    tr_mask, te_mask = df["is_train"] == 1, df["is_train"] == 0
    tr = df[tr_mask & df[col].notna()].copy()
    yb = (tr[TARGET].values > 0.5).astype(int)
    for enc, val in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(tr, yb):
        s = tr.iloc[enc].groupby(col)[TARGET].agg(["mean", "count"])
        m = (s["mean"] * s["count"] + gmean * smoothing) / (s["count"] + smoothing)
        df.loc[tr.index[val], f"{col}_te"] = df.loc[tr.index[val], col].map(m)
    s = tr.groupby(col)[TARGET].agg(["mean", "count"])
    m = (s["mean"] * s["count"] + gmean * smoothing) / (s["count"] + smoothing)
    df.loc[te_mask, f"{col}_te"] = df.loc[te_mask, col].map(m)
    df[f"{col}_te"] = df[f"{col}_te"].fillna(gmean)
    return f"{col}_te"

te = [target_encode_cv(c, s) for c, s in [("fl_adminarea", 30), ("db_group_last", 20), ("dt_month", 30), ("cb_rate", 5)]]
te = [f for f in te if df[f].max() - df[f].min() > 0.005]  # отбрасываем вырожденные
te

In [ ]:
# два набора признаков: без target encoding (v1) и с ним (v3)
base = ["loan_amount_last", "overdraft_limit_min", "overdraft_limit_max", "offered_rate", "cb_rate",
        "corp_credit_products", "sum_deb_ul_90", "cnt_deb_loan_90", "cnt_deb_ul_ip_90", "cnt_deb_ul_ip_30",
        "balance_rur_amt_30_min", "cnt_cred_loan_90", "loan_rev_max_start_non_fin", "loan_rev_min_start_fin",
        "app_term_mean_360", "overdraft_app_term_max_360", "days_from_authperson_registration",
        "fl_hdb_bki_total_active_products", "corp_list", "count_all_corp_dashboard_events",
        "p75_time_spent_minutes", "sum_deb_investment_90"]
reserved = base + CAT + date_feats + miss + inter + te + [ID_COL, TARGET, "is_train"]
extra = [c for c in df.columns if c not in reserved and df[c].dtype in [np.float64, np.int64]]
base = [c for c in set(base + extra) if c in df.columns]

feats_v1 = [f for f in dict.fromkeys(base + CAT + date_feats + miss + inter) if f in df.columns]
feats_v3 = list(dict.fromkeys(feats_v1 + te))

train_df = df[df["is_train"] == 1].reset_index(drop=True)
test_df = df[df["is_train"] == 0].reset_index(drop=True)
X1tr, X1te = train_df[feats_v1], test_df[feats_v1]
X3tr, X3te = train_df[feats_v3], test_df[feats_v3]
spw = (y == 0).sum() / (y == 1).sum()  # дисбаланс классов
print(f"признаков: v1={len(feats_v1)}, v3={len(feats_v3)}")

## Валидация

5-fold **stratified**. Хронологическую и оконную валидацию пробовали — на лидерборде
они оказались хуже, т.к. оставляют меньше данных на обучение в каждом фолде.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def run_lgb(params, Xtr, Xte, rounds, stop):
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for tr, va in skf.split(Xtr, y):
        d_tr = lgb.Dataset(Xtr.iloc[tr], label=y.iloc[tr], categorical_feature=CAT)
        d_va = lgb.Dataset(Xtr.iloc[va], label=y.iloc[va], categorical_feature=CAT)
        m = lgb.train(params, d_tr, num_boost_round=rounds, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(stop), lgb.log_evaluation(0)])
        oof[va] = m.predict(Xtr.iloc[va]); tst += m.predict(Xte) / N_FOLDS
    return oof, tst

def run_cb(Xtr, Xte, off=0):
    idx = [list(Xtr.columns).index(c) for c in CAT]
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for fold, (tr, va) in enumerate(skf.split(Xtr, y)):
        m = cb.CatBoostClassifier(iterations=10000, learning_rate=0.02, depth=8, l2_leaf_reg=5,
                                  min_data_in_leaf=50, eval_metric="AUC", loss_function="Logloss",
                                  auto_class_weights="Balanced", random_seed=SEED + fold + off,
                                  verbose=0, early_stopping_rounds=200, allow_writing_files=False)
        m.fit(cb.Pool(Xtr.iloc[tr], y.iloc[tr], cat_features=idx),
              eval_set=cb.Pool(Xtr.iloc[va], y.iloc[va], cat_features=idx), use_best_model=True)
        oof[va] = m.predict_proba(Xtr.iloc[va])[:, 1]; tst += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, tst

def run_xgb(params, Xtr, Xte, rounds, stop):
    Xtr, Xte = Xtr.copy(), Xte.copy()
    for c in CAT:
        Xtr[c], Xte[c] = Xtr[c].astype("category"), Xte[c].astype("category")
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for tr, va in skf.split(Xtr, y):
        d_tr = xgb.DMatrix(Xtr.iloc[tr], label=y.iloc[tr], enable_categorical=True)
        d_va = xgb.DMatrix(Xtr.iloc[va], label=y.iloc[va], enable_categorical=True)
        m = xgb.train(params, d_tr, num_boost_round=rounds, evals=[(d_va, "v")],
                      early_stopping_rounds=stop, verbose_eval=0)
        oof[va] = m.predict(d_va); tst += m.predict(xgb.DMatrix(Xte, enable_categorical=True)) / N_FOLDS
    return oof, tst

## Вариант 1 — одиночная модель (бейзлайн)

Одиночный CatBoost даёт около 0.834 OOF — отправная точка.

In [ ]:
oof_cb, _ = run_cb(X3tr, X3te)
print("одиночный CatBoost OOF AUC:", round(roc_auc_score(y, oof_cb), 5))

## Вариант 2 — ансамбль 7 моделей

Три типа бустинга на двух наборах признаков. Базовые (lr=0.02) и консервативные
(lr=0.01, сильнее регуляризация — лучшие одиночные). Финальное предсказание —
простое среднее: взвешенное и стекинг переобучались (корреляция моделей 0.95+).

In [ ]:
def params(off=0):
    lgb1 = dict(objective="binary", metric="auc", learning_rate=0.02, num_leaves=63, min_child_samples=50,
                feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=5, reg_alpha=0.3, reg_lambda=0.3,
                min_gain_to_split=0.01, scale_pos_weight=spw, verbose=-1, n_jobs=-1, random_state=SEED + off)
    lgb4 = dict(objective="binary", metric="auc", learning_rate=0.01, num_leaves=31, min_child_samples=80,
                feature_fraction=0.5, bagging_fraction=0.6, bagging_freq=5, reg_alpha=1.0, reg_lambda=1.0,
                min_gain_to_split=0.02, scale_pos_weight=spw, verbose=-1, n_jobs=-1, random_state=SEED + 100 + off)
    xgb1 = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.02, max_depth=7,
                min_child_weight=50, subsample=0.7, colsample_bytree=0.65, reg_alpha=0.3, reg_lambda=0.3,
                scale_pos_weight=spw, tree_method="hist", random_state=SEED + off, verbosity=0, enable_categorical=True)
    xgb2 = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.01, max_depth=5,
                min_child_weight=80, subsample=0.6, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=1.0,
                scale_pos_weight=spw, tree_method="hist", random_state=SEED + 300 + off, verbosity=0, enable_categorical=True)
    return lgb1, lgb4, xgb1, xgb2

def ensemble(off=0):
    lgb1, lgb4, xgb1, xgb2 = params(off)
    oofs, tsts = [], []
    for o, t in [run_lgb(lgb1, X1tr, X1te, 10000, 200),
                 run_cb(X1tr, X1te, off),
                 run_lgb(lgb1, X3tr, X3te, 10000, 200),
                 run_cb(X3tr, X3te, off),
                 run_xgb(xgb1, X3tr, X3te, 10000, 200),
                 run_lgb(lgb4, X3tr, X3te, 15000, 300),
                 run_xgb(xgb2, X3tr, X3te, 15000, 300)]:
        oofs.append(o); tsts.append(t)
    return np.mean(oofs, axis=0), np.mean(tsts, axis=0)

oof2, test2 = ensemble()
print("ансамбль 7 моделей OOF AUC:", round(roc_auc_score(y, oof2), 5))

## Вариант 3 — ансамбль + seed-bagging (финал)

Повторяем ансамбль с несколькими сидами разбиения и усредняем. Снижает дисперсию
случайного фолда, дало финальный прирост на лидерборде.

In [ ]:
BAG_SEEDS = [42, 202, 707]  # больше сидов — убывающая отдача
oofs, tests = [], []
for s in BAG_SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=s)
    o, t = ensemble(off=s - SEED)
    oofs.append(o); tests.append(t)
    print(f"seed {s}: OOF {roc_auc_score(y, o):.5f}")

oof_final = np.mean(oofs, axis=0)
test_final = np.mean(tests, axis=0)
print("seed-bagged OOF AUC:", round(roc_auc_score(y, oof_final), 5))

## Сабмит

In [ ]:
submission = pd.DataFrame({"front_id": test_df[ID_COL].astype(int).values, "target_value": test_final})
submission.to_csv("submission.csv", index=False)
submission.head()

## Что ещё пробовали

Коротко — что проверяли и почему не вошло в финал:

- **хронологическая / оконная валидация** — ниже на лидерборде (меньше данных на фолд);
- **recency-взвешивание** свежих наблюдений — на лидерборде без прироста;
- **стекинг и оптимизация весов** ансамбля — переобучались при корреляции моделей 0.95–0.99;
- **псевдоразметка** уверенных предсказаний теста — прирост в пределах шума;
- **добавление ExtraTrees/DART** — чуть лучше на OOF, но на лидерборде не подтвердилось;
- **модели на подгруппах признаков** — упало индивидуальное качество.